In [ ]:
#!/usr/bin/env python3
"""
RSL Entry GUI Application
A tkinter-based GUI for entering RSL (Relevant/SL) data.
Saves entries to RSL_Entry_mm.csv
"""

import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime
import csv
import os


class RSLEntryGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("RSL Entry System")
        self.root.geometry("950x750")
        
        # Set style
        style = ttk.Style()
        style.theme_use('clam')
        style.configure('TLabel', font=('Arial', 10))
        style.configure('Header.TLabel', font=('Arial', 11, 'bold'))
        
        # Create main frame
        main_frame = ttk.Frame(root, padding="15")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Configure grid weights for responsiveness
        root.columnconfigure(0, weight=1)
        root.rowconfigure(0, weight=1)
        
        # ===== Personal Information Section =====
        ttk.Label(main_frame, text="📋 Personal Information", style='Header.TLabel').grid(
            row=0, column=0, columnspan=4, pady=(10, 10), sticky=tk.W)
        
        # Name
        ttk.Label(main_frame, text="Name:").grid(row=1, column=0, sticky=tk.W, pady=5, padx=5)
        self.name_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.name_var, width=45).grid(
            row=1, column=1, columnspan=3, sticky=tk.W, pady=5, padx=5)
        
        # DoB
        ttk.Label(main_frame, text="DoB (YYYY-MM-DD):").grid(row=2, column=0, sticky=tk.W, pady=5, padx=5)
        self.dob_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.dob_var, width=22).grid(
            row=2, column=1, sticky=tk.W, pady=5, padx=5)
        
        # Gender
        ttk.Label(main_frame, text="Gender:").grid(row=2, column=2, sticky=tk.W, pady=5, padx=(20, 5))
        self.gender_var = tk.StringVar()
        gender_combo = ttk.Combobox(main_frame, textvariable=self.gender_var, width=20, state='readonly')
        gender_combo['values'] = ('Male', 'Female', 'Other')
        gender_combo.grid(row=2, column=3, sticky=tk.W, pady=5, padx=5)
        
        # NRC
        ttk.Label(main_frame, text="NRC:").grid(row=3, column=0, sticky=tk.W, pady=5, padx=5)
        self.nrc_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.nrc_var, width=22).grid(
            row=3, column=1, sticky=tk.W, pady=5, padx=5)
        
        # Phone
        ttk.Label(main_frame, text="Phone:").grid(row=3, column=2, sticky=tk.W, pady=5, padx=(20, 5))
        self.phone_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.phone_var, width=20).grid(
            row=3, column=3, sticky=tk.W, pady=5, padx=5)
        
        # Assignor
        ttk.Label(main_frame, text="Assignor:").grid(row=4, column=0, sticky=tk.W, pady=5, padx=5)
        self.assignor_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.assignor_var, width=22).grid(
            row=4, column=1, sticky=tk.W, pady=5, padx=5)
        
        # Type
        ttk.Label(main_frame, text="Type:").grid(row=4, column=2, sticky=tk.W, pady=5, padx=(20, 5))
        self.type_var = tk.StringVar()
        type_combo = ttk.Combobox(main_frame, textvariable=self.type_var, width=20, state='readonly')
        type_combo['values'] = ('Stu', 'Ds')
        type_combo.grid(row=4, column=3, sticky=tk.W, pady=5, padx=5)
        
        # Separator
        ttk.Separator(main_frame, orient='horizontal').grid(
            row=5, column=0, columnspan=4, sticky='ew', pady=15)
        
        # ===== Choose Section (SL/RL) =====
        ttk.Label(main_frame, text="🎯 Choose Type", style='Header.TLabel').grid(
            row=6, column=0, columnspan=4, pady=(10, 10), sticky=tk.W)
        
        self.choose_var = tk.StringVar(value="SL")
        
        choose_frame = ttk.Frame(main_frame)
        choose_frame.grid(row=7, column=0, columnspan=4, sticky=tk.W, padx=5)
        
        sl_radio = ttk.Radiobutton(choose_frame, text="SL", variable=self.choose_var, 
                                   value="SL", command=self.toggle_code_options)
        sl_radio.grid(row=0, column=0, padx=5)
        
        rl_radio = ttk.Radiobutton(choose_frame, text="RL", variable=self.choose_var, 
                                   value="RL", command=self.toggle_code_options)
        rl_radio.grid(row=0, column=1, padx=5)
        
        # ===== SL Code Section =====
        sl_frame = ttk.LabelFrame(main_frame, text="SL Code Options", padding="10")
        sl_frame.grid(row=8, column=0, columnspan=4, sticky=tk.W, pady=10, padx=5)
        
        ttk.Label(sl_frame, text="SL Code:").grid(row=0, column=0, sticky=tk.W, pady=5)
        self.sl_code_var = tk.StringVar()
        self.sl_code_combo = ttk.Combobox(sl_frame, textvariable=self.sl_code_var, 
                                          width=45, state='readonly')
        self.sl_code_combo['values'] = ('IR', 'MT', 'TV', 'W1yr', 'W2yr', 'W5yr')
        self.sl_code_combo.grid(row=0, column=1, sticky=tk.W, pady=5, padx=10)
        
        # SL Code descriptions
        self.sl_descriptions = {
            'IR': 'Instant Refusal',
            'MT': 'Mixing Techniques on Course',
            'TV': 'Teaching Vipassana',
            'W1yr': 'Wait 1 year',
            'W2yr': 'Wait 2 years',
            'W5yr': 'Wait 5 years'
        }
        
        self.sl_desc_label = ttk.Label(sl_frame, text="", foreground='blue', font=('Arial', 10, 'italic'))
        self.sl_desc_label.grid(row=1, column=1, sticky=tk.W, pady=2, padx=10)
        self.sl_code_combo.bind('<<ComboboxSelected>>', self.update_sl_description)
        
        # ===== RL Code Section =====
        rl_frame = ttk.LabelFrame(main_frame, text="RL Code Options", padding="10")
        rl_frame.grid(row=9, column=0, columnspan=4, sticky=tk.W, pady=10, padx=5)
        
        ttk.Label(rl_frame, text="RL Code:").grid(row=0, column=0, sticky=tk.W, pady=5)
        self.rl_code_var = tk.StringVar()
        self.rl_code_combo = ttk.Combobox(rl_frame, textvariable=self.rl_code_var, 
                                          width=45, state='readonly')
        self.rl_code_combo['values'] = ('DRI', 'IP', 'ML', 'NC', 'NI', 'RefAT', 
                                        'SBSv', 'SCN', 'TOT', 'W6m', 'W12m', 'W18m')
        self.rl_code_combo.grid(row=0, column=1, sticky=tk.W, pady=5, padx=10)
        
        # RL Code descriptions
        self.rl_descriptions = {
            'DRI': 'Denial of Relevant Info',
            'IP': 'Incompatible Practice',
            'ML': 'Medical Letter Required',
            'NC': 'Non compliance',
            'NI': 'Need Interpreter',
            'RefAT': 'Refer to AT',
            'SBSv': 'Sit before Serve',
            'SCN': 'Strong Commitment Needed',
            'TOT': 'Teaching Other Techniques',
            'W6m': 'Wait 6 months',
            'W12m': 'Wait 12 months',
            'W18m': 'Wait 18 months'
        }
        
        self.rl_desc_label = ttk.Label(rl_frame, text="", foreground='blue', font=('Arial', 10, 'italic'))
        self.rl_desc_label.grid(row=1, column=1, sticky=tk.W, pady=2, padx=10)
        self.rl_code_combo.bind('<<ComboboxSelected>>', self.update_rl_description)
        
        # ===== Course Information Section =====
        ttk.Label(main_frame, text="📚 Course Information", style='Header.TLabel').grid(
            row=10, column=0, columnspan=4, pady=(20, 10), sticky=tk.W)
        
        # CourseType
        ttk.Label(main_frame, text="Course Type:").grid(row=11, column=0, sticky=tk.W, pady=5, padx=5)
        self.course_type_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.course_type_var, width=45).grid(
            row=11, column=1, columnspan=3, sticky=tk.W, pady=5, padx=5)
        
        # CourseDate
        ttk.Label(main_frame, text="Course Date (YYYY-MM-DD):").grid(row=12, column=0, sticky=tk.W, pady=5, padx=5)
        self.course_date_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.course_date_var, width=22).grid(
            row=12, column=1, sticky=tk.W, pady=5, padx=5)
        
        # ReAdmitDate
        ttk.Label(main_frame, text="Re-Admit Date (YYYY-MM-DD):").grid(row=12, column=2, 
                                                                       sticky=tk.W, pady=5, padx=(20, 5))
        self.readmit_date_var = tk.StringVar()
        ttk.Entry(main_frame, textvariable=self.readmit_date_var, width=20).grid(
            row=12, column=3, sticky=tk.W, pady=5, padx=5)
        
        # ===== Buttons Section =====
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=13, column=0, columnspan=4, pady=(30, 10))
        
        save_btn = ttk.Button(button_frame, text="💾 Save Entry", command=self.save_entry, width=20)
        save_btn.grid(row=0, column=0, padx=10)
        
        clear_btn = ttk.Button(button_frame, text="🔄 Clear Form", command=self.clear_form, width=20)
        clear_btn.grid(row=0, column=1, padx=10)
        
        # Initialize
        self.toggle_code_options()
        
    def toggle_code_options(self):
        """Toggle between SL and RL code visibility"""
        if self.choose_var.get() == "SL":
            self.sl_code_combo.config(state='readonly')
            self.rl_code_combo.config(state='disabled')
            self.sl_desc_label.config(text=self.sl_descriptions.get(self.sl_code_var.get(), ""))
            self.rl_desc_label.config(text="")
        else:
            self.sl_code_combo.config(state='disabled')
            self.rl_code_combo.config(state='readonly')
            self.sl_desc_label.config(text="")
            self.rl_desc_label.config(text=self.rl_descriptions.get(self.rl_code_var.get(), ""))
    
    def update_sl_description(self, event=None):
        """Update SL description label"""
        code = self.sl_code_var.get()
        self.sl_desc_label.config(text=f"→ {self.sl_descriptions.get(code, '')}")
    
    def update_rl_description(self, event=None):
        """Update RL description label"""
        code = self.rl_code_var.get()
        self.rl_desc_label.config(text=f"→ {self.rl_descriptions.get(code, '')}")
    
    def validate_date(self, date_str, field_name):
        """Validate date format YYYY-MM-DD"""
        if not date_str:
            return True  # Allow empty dates
        try:
            datetime.strptime(date_str, '%Y-%m-%d')
            return True
        except ValueError:
            messagebox.showerror("Invalid Date", f"{field_name} must be in YYYY-MM-DD format")
            return False
    
    def save_entry(self):
        """Save entry to CSV file"""
        # Validate dates
        if not self.validate_date(self.dob_var.get(), "DoB"):
            return
        if not self.validate_date(self.course_date_var.get(), "Course Date"):
            return
        if not self.validate_date(self.readmit_date_var.get(), "Re-Admit Date"):
            return
        
        # Get the chosen type
        choose_type = self.choose_var.get()
        if choose_type == "SL":
            code = self.sl_code_var.get()
            code_desc = self.sl_descriptions.get(code, "")
        else:
            code = self.rl_code_var.get()
            code_desc = self.rl_descriptions.get(code, "")
        
        # Prepare data
        data = {
            'Name': self.name_var.get(),
            'DoB': self.dob_var.get(),
            'Gender': self.gender_var.get(),
            'NRC': self.nrc_var.get(),
            'Phone': self.phone_var.get(),
            'Assignor': self.assignor_var.get(),
            'Type': self.type_var.get(),
            'S_L': choose_type,
            'Code': code,
            'Code_Description': code_desc,
            'CourseType': self.course_type_var.get(),
            'CourseDate': self.course_date_var.get(),
            'ReAdmitDate': self.readmit_date_var.get()
        }
        
        # Define CSV headers
        headers = ['Name', 'DoB', 'Gender', 'NRC', 'Phone', 'Assignor', 'Type', 
                   'S_L', 'Code', 'Code_Description', 'CourseType', 'CourseDate', 'ReAdmitDate']
        
        # Check if file exists to determine if we need to write headers
        file_exists = os.path.exists('RSL_Entry_mm.csv')
        
        try:
            with open('RSL_Entry_mm.csv', 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=headers)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(data)
            
            messagebox.showinfo("Success", "Entry saved successfully to RSL_Entry_mm.csv!")
            self.clear_form()
        except Exception as e:
            messagebox.showerror("Error", f"Failed to save entry: {str(e)}")
    
    def clear_form(self):
        """Clear all form fields"""
        self.name_var.set('')
        self.dob_var.set('')
        self.gender_var.set('')
        self.nrc_var.set('')
        self.phone_var.set('')
        self.assignor_var.set('')
        self.type_var.set('')
        self.choose_var.set('SL')
        self.sl_code_var.set('')
        self.rl_code_var.set('')
        self.course_type_var.set('')
        self.course_date_var.set('')
        self.readmit_date_var.set('')
        self.toggle_code_options()


def main():
    root = tk.Tk()
    
    # Set window icon (optional)
    try:
        root.iconbitmap('@icon.xbm')
    except:
        pass
    
    app = RSLEntryGUI(root)
    root.mainloop()


if __name__ == "__main__":
    main()

: 